# Decibel P1 TrueVoice asset export

This notebook runs the prepared P1 exporter on a temporary Google Colab runtime. It downloads the 16 GB Gemma parent checkpoint to the runtime disk, loads only the Gemma audio tower, checks a synthetic inference, and packages the tower, processor, classifier head, and checksummed manifest. It never mounts Google Drive and does not download Gemma weights to your computer.

Use a free GPU runtime. The preflight requires at least 8 GiB GPU VRAM, 8 GiB available system RAM, and 24 GiB free runtime disk. Free Colab hardware is not guaranteed.

Run each cell from top to bottom. The upload cell asks for exactly `scripts/export_truevoice_assets.py` and `saved_model/classifier_head.pt`. The final cell creates a browser download for `/content/decibel_truevoice_assets.zip`.

In [ ]:
# Preflight before installing packages or contacting Hugging Face.
import shutil, torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab select Runtime > Change runtime type > T4 GPU, reconnect, then rerun this cell.')
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 2**30
memory = {}
with open('/proc/meminfo', encoding='ascii') as f:
    for line in f:
        key, value = line.split(':', 1)
        if key in {'MemTotal', 'MemAvailable'}:
            memory[key] = int(value.split()[0]) / 2**20
free_disk_gib = shutil.disk_usage('/content').free / 2**30
print(f'GPU: {gpu.name} | VRAM: {gpu_gib:.1f} GiB')
print(f'System RAM: {memory["MemTotal"]:.1f} GiB total | {memory["MemAvailable"]:.1f} GiB available')
print(f'Colab disk: {free_disk_gib:.1f} GiB free')
if gpu_gib < 8 or memory['MemAvailable'] < 8 or free_disk_gib < 24:
    raise RuntimeError('Preflight failed: need >=8 GiB GPU VRAM, >=8 GiB available RAM, and >=24 GiB free disk. No model weights have been requested.')
print('Preflight passed. No model weights have been requested yet.')

In [ ]:
# Pin exporter dependencies. Keep Colab's CUDA-enabled PyTorch installation unchanged.
%pip -q install transformers==5.18.0 accelerate==1.15.0 librosa==0.11.0 safetensors==0.8.0 huggingface-hub==1.33.0
import importlib.metadata as md, torch
expected = {'transformers': '5.18.0', 'accelerate': '1.15.0', 'librosa': '0.11.0', 'safetensors': '0.8.0', 'huggingface-hub': '1.33.0'}
for package, wanted in expected.items():
    actual = md.version(package)
    print(f'{package}=={actual}')
    if actual != wanted:
        raise RuntimeError(f'Expected {package}=={wanted}, found {actual}')
if tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) < (2, 5):
    raise RuntimeError(f'PyTorch >=2.5 is required; Colab has {torch.__version__}')
print(f'torch=={torch.__version__} (Colab-provided build retained)')

In [ ]:
# Upload the P1 exporter and the classifier checkpoint; do not upload metadata.json.
from google.colab import files
from pathlib import Path
uploaded = files.upload()
required = {'export_truevoice_assets.py', 'classifier_head.pt'}
missing = required.difference(uploaded)
if missing:
    raise RuntimeError(f'Missing upload(s): {sorted(missing)}. Upload the exporter script and classifier_head.pt.')
for filename, content in uploaded.items():
    Path('/content', filename).write_bytes(content)
print('Both required files are ready in /content.')

In [ ]:
# Hugging Face is tried without credentials first. If the Hub actually returns
# 401/403, getpass asks for a token without echoing it; it is process-local only.
import os
from getpass import getpass
import export_truevoice_assets as exporter

def _is_hf_auth_error(exc):
    response = getattr(exc, 'response', None)
    status = getattr(response, 'status_code', None)
    if status in (401, 403):
        return True
    message = str(exc).lower()
    return ('huggingface.co' in message and ('401 client error' in message or '403 client error' in message or 'gatedrepoerror' in message))

output_zip = Path('/content/decibel_truevoice_assets.zip')
try:
    exporter.export_assets(Path('/content/classifier_head.pt'), output_zip)
except Exception as exc:
    if not _is_hf_auth_error(exc):
        raise
    print('Hugging Face requires authentication for this model request. Enter a read token in the hidden prompt; it will not be printed or saved.')
    previous_token = os.environ.get('HF_TOKEN')
    entered_token = getpass('Hugging Face token (hidden): ').strip()
    if not entered_token:
        raise RuntimeError('No token entered; export stopped.') from None
    os.environ['HF_TOKEN'] = entered_token
    entered_token = ''
    try:
        exporter.export_assets(Path('/content/classifier_head.pt'), output_zip)
    finally:
        if previous_token is None:
            os.environ.pop('HF_TOKEN', None)
        else:
            os.environ['HF_TOKEN'] = previous_token
        previous_token = None

if not output_zip.is_file():
    raise RuntimeError('Export finished without creating the expected ZIP.')
print(f'EXPORT COMPLETE: {output_zip} ({output_zip.stat().st_size / 2**30:.2f} GiB)')
print('Starting the Colab browser download. Save the downloaded ZIP; it contains the exported tower, processor, head, and manifest.')
files.download(str(output_zip))